In [50]:
import os, json
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.document_loaders import JSONLoader
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

load_dotenv()
LLM = ChatGroq(model="llama-3.3-70b-versatile", api_key=os.getenv("GROQ_API_KEY"))
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights: 100%|██████████| 103/103 [00:01<00:00, 77.12it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [51]:
# Load & format docs 
def format_transaction(record: dict, metadata: dict) -> dict:
    metadata["id"] = record.get("id")
    metadata["status"] = record.get("status")
    metadata["type"] = record.get("type")
    metadata["amount"] = record.get("amount")
    metadata["name"] = record.get("name")
    return metadata

loader = JSONLoader(
    file_path="src/data/transactions.json",   # use forward slashes
    jq_schema=".[]",
    text_content=False,
    metadata_func=format_transaction
)
docs = loader.load()

# Rewrite page_content to human-readable text
for doc in docs:
    data = json.loads(doc.page_content)
    direction = "Sent to" if data.get("type") == "debit" else "Received from"
    doc.page_content = (
        f"Transaction ID: {data.get('id')} | "
        f"{direction} {data.get('name')} ({data.get('upiId')}) | "
        f"Amount: ₹{data.get('amount')} | "
        f"Status: {data.get('status')} | "
        f"Method: {data.get('paymentMethod')} | "
        f"Date: {data.get('date')} | "
        f"Note: {data.get('note')} | "
        f"Bank Ref: {data.get('bankRef')}"
    )


In [52]:
# Vectorstore 
vectorstore = FAISS.from_documents(docs, embedding)
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 20, "fetch_k": 30}
)

In [54]:
# Prompt & Chain
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are PayBot, a transaction assistant for a PhonePe-style payment app.
Answer ONLY from the context provided. Do not guess or hallucinate.
Use ₹ for amounts. Keep answers short and clear.

<context>
{context}
</context>"""),
("human", "{input}")
])

document_chain = create_stuff_documents_chain(LLM, prompt)
retrieval_chain = create_retrieval_chain(retriever, document_chain)

In [56]:
response = retrieval_chain.invoke({"input": "What is the total amount i sent to Karthik overall?"})
print(response["answer"])

To find the total amount sent to Karthik, I'll add up the amounts from the 'Sent to Karthik' transactions: 
₹600 + ₹4500 + ₹1500 + ₹300 + ₹1800 + ₹950 + ₹560 + ₹275 + ₹3000 + ₹2950 + ₹640 + ₹1450 + ₹340 + ₹1800 + ₹220 = ₹16,130. 

However, note that two of these transactions (₹4500 and ₹3000) are still 'pending'. If we only consider the 'success' transactions, the total amount sent to Karthik is: 
₹600 + ₹1500 + ₹300 + ₹1800 + ₹950 + ₹560 + ₹275 + ₹2950 + ₹640 + ₹1450 + ₹340 + ₹1800 + ₹220 = ₹11,630.
